In [ ]:
import re
from pathlib import Path

import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

In [ ]:
base_path = Path("../../../processed_data/nbs-river-catchment")

In [ ]:
# Lookup table from ensemble member ID to cost/damage sensitivity
# parameters

ensemble_metadata = pd.read_csv(base_path / "sensitivity_parameters.csv")
ensemble_metadata

In [ ]:
# CRS and affine transform parameters for each hazard raster file
# The affine transform parameters can be used to transform the grid
# indices in the damage files into lat/long

raster_metadata = pd.read_csv(base_path / "_hazard_layers__with_transforms.csv")
raster_metadata.query('hazard == "fluvial" and rp == 1500 and rcp == "baseline"')

In [ ]:
raster_metadata.transform_id.unique()

In [ ]:
# Example damage file
def read_damage_file(fname):
    fname = str(fname)
    if "_nodes" in fname:
        id_columns = ["node_id"]
    if "_edges" in fname:
        id_columns = ["edge_id"]
    if "_areas" in fname:
        id_columns = ["node_id"]
    if "electricity_network_v3.1_nodes" in fname:
        id_columns = ["id"]
    if "buildings" in fname:
        id_columns = ["osm_id"]

    columns = id_columns + [
        "exposure_unit",
        "damage_cost_unit",
        "exposure",
        "cell_index_2_x",
        "cell_index_2_y",
        "damage_uncertainty_parameter",
        "cost_uncertainty_parameter",
        "fluvial__rp_20__rcp_baseline__epoch_2010__conf_None",
        "fluvial__rp_50__rcp_baseline__epoch_2010__conf_None",
        "fluvial__rp_100__rcp_baseline__epoch_2010__conf_None",
        "fluvial__rp_200__rcp_baseline__epoch_2010__conf_None",
        "fluvial__rp_500__rcp_baseline__epoch_2010__conf_None",
        "fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None"
    ]
    example_damage = pd.read_parquet(
        fname,
        columns=columns,
        filters=[('fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None', '>', 0)]
    ).rename(columns={
        id_col: "asset_id" for id_col in id_columns
    })

    return example_damage

example_damage = read_damage_file(base_path / "direct_damages"/ "rail_edges" / "rail_edges_direct_damages_parameter_set_2.parquet").query('fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None > 0')
example_damage.columns, example_damage.shape

In [ ]:
damage_paths = list(Path(base_path / "direct_damages").glob("**/*.parquet"))
damage_paths

In [ ]:
# How to read every damage file in a loop, and pick out
# its asset class from the filename -- when you need to...

damage_paths = list(Path(base_path / "direct_damages").glob("**/*.parquet"))

dfs = []
for path in tqdm(damage_paths):
    match = re.match(
        r"([A-Za-z0-9_.]+)_direct_damages_parameter_set_(\d+)\.parquet",
        Path(path).name
    )
    asset_class, ensemble_member = match.groups()
    if "electricity_network_v3.1_edges" in str(path):
        continue

    try:
        df = read_damage_file(path)
    except Exception as e:
        print(path)
        raise e
    df['asset_class'] = asset_class
    df['ensemble_member'] = ensemble_member
    dfs.append(df)

all_damage = pd.concat(dfs)

In [ ]:
all_damage_minmax = all_damage.query("(damage_uncertainty_parameter == 0 and cost_uncertainty_parameter == 0) or (damage_uncertainty_parameter == 1 and cost_uncertainty_parameter == 1)").copy()

In [ ]:
all_damage_minmax.dtypes#to_parquet(base_path / "all_damage_minmax.parquet", index=False)

In [ ]:
all_damage_minmax.asset_id = all_damage_minmax.asset_id.astype("str")

In [ ]:
all_damage_minmax.to_parquet(base_path / "all_damage_minmax.parquet", index=False)

In [ ]:
# read non snapped points
non_snapped_points_with_flood = gpd.read_parquet(base_path / "non_snapped_points_with_flood_indices.parquet").reset_index().rename(columns={"index":"fid"})
non_snapped_points_with_flood

In [ ]:
# Keep all points (right), match damage on its cell_index_2_* to points’ flood_i/j
merged_non_snapped_points_with_flood_damage_df = all_damage_minmax.merge(
    non_snapped_points_with_flood,
    how="left",
    left_on=["cell_index_2_x", "cell_index_2_y"],
    right_on=["flood_i", "flood_j"],
)
unique_damage_fids = merged_non_snapped_points_with_flood_damage_df.dropna().fid.unique()
merged_non_snapped_points_with_flood_damage_df.head(2)

In [ ]:
snapped_points_with_dem = base_path / "snapped_points_with_dem_indices.parquet"
snapped_points_with_dem = gpd.read_parquet(snapped_points_with_dem).reset_index().rename(columns={"index":"fid"})
snapped_points_with_dem

In [ ]:
merged_fid = non_snapped_points_with_flood.merge(
    snapped_points_with_dem,
    on="fid",
    how="left",
    validate="many_to_one"
)

In [ ]:
flood_ij_to_dem_ij = merged_fid.loc[sorted(unique_damage_fids)][["flood_i", "flood_j", "dem_i", "dem_j"]].reset_index(drop=True)
flood_ij_to_dem_ij.to_parquet(base_path / "flood_ij_to_dem_ij_relation.parquet")
flood_ij_to_dem_ij

In [ ]:
points_to_catchment_df = merged_fid.loc[sorted(unique_damage_fids)][["geometry_y", "dem_i", "dem_j"]].drop_duplicates(subset=["dem_i","dem_j"])
points_to_catchment_df.rename(columns={'geometry_y':'geometry'}, inplace=True)
points_to_catchment_df.to_parquet(base_path / "points_to_catchment.parquet", index=False)
points_to_catchment_df